In [12]:
import os
import datetime
import pdfplumber
import pandas as pd

try:
    from tabulate import tabulate
    use_tabulate = True
except ImportError:
    use_tabulate = False

folder_path = r"Z:\HTOC\HTOC Reports\I&W Reports\3. Finished I&W Reports"
two_weeks_ago = datetime.datetime.now() - datetime.timedelta(weeks=2)

# Find recent PDFs
recent_pdfs = []
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if os.path.isfile(file_path) and filename.lower().endswith('.pdf'):
        modified_time = datetime.datetime.fromtimestamp(os.path.getmtime(file_path))
        if modified_time >= two_weeks_ago:
            recent_pdfs.append(file_path)

if not recent_pdfs:
    print("No recent PDF files found.")
else:
    all_pdf_data = {}  # Store data per PDF

    for pdf_file in recent_pdfs:
        print(f"\nProcessing PDF: {pdf_file}\n")
        pdf_data = {
            "title_serial_paragraph": None,
            "tables": []
        }

        with pdfplumber.open(pdf_file) as pdf:
            first_page = pdf.pages[0]
            full_text = first_page.extract_text()

            if full_text:
                exec_summary_idx = full_text.find("Executive Summary:")
                if exec_summary_idx != -1:
                    full_text = full_text[:exec_summary_idx].strip()

                lines = [line.strip() for line in full_text.split('\n')]

                idx = next((i for i, line in enumerate(lines) if "Title/Serial Number:" in line), None)

                if idx is not None:
                    line_after_title = lines[idx].split("Title/Serial Number:", 1)[1].strip()
                    collected_lines = [line_after_title] if line_after_title else []

                    for line in lines[idx+1:]:
                        if line == '':
                            break
                        collected_lines.append(line)

                    title_serial_paragraph = '\n'.join(collected_lines)
                    pdf_data["title_serial_paragraph"] = title_serial_paragraph
                else:
                    print(f"'Title/Serial Number:' not found in the text of {os.path.basename(pdf_file)}.")
            else:
                print(f"No text found on the first page of {os.path.basename(pdf_file)}.")

            # Extract tables from all pages
            for page_num, page in enumerate(pdf.pages, start=1):
                tables = page.extract_tables()
                if not tables:
                    continue
                for idx, table in enumerate(tables):
                    df = pd.DataFrame(table[1:], columns=table[0])

                    # Fix split Indicator Type columns if present
                    if 'Indicator' in df.columns and 'Type' in df.columns:
                        df['Indicator'].replace('', pd.NA, inplace=True)
                        df['Type'].replace('', pd.NA, inplace=True)
                        df['Indicator'] = df['Indicator'].ffill()
                        df['Type'] = df['Type'].ffill()
                        df['Indicator Type'] = (df['Indicator'].astype(str) + ' ' + df['Type'].astype(str)).str.strip()
                        df.drop(columns=['Indicator', 'Type'], inplace=True)

                    pdf_data["tables"].append({
                        "page": page_num,
                        "table_index": idx + 1,
                        "dataframe": df
                    })

        all_pdf_data[os.path.basename(pdf_file)] = pdf_data

    # Now print out the extracted data per PDF
    for pdf_name, data in all_pdf_data.items():
        print(f"\n--- Data from PDF: {pdf_name} ---\n")

        if data["title_serial_paragraph"]:
            print("Title/Serial Number Paragraph:\n")
            print(data["title_serial_paragraph"])
        else:
            print("No Title/Serial Number paragraph extracted.")

        if data["tables"]:
            print(f"\nExtracted {len(data['tables'])} table(s):")
            for table_info in data["tables"]:
                print(f"\nPage {table_info['page']} - Table {table_info['table_index']}:\n")
                df = table_info["dataframe"]
                if use_tabulate:
                    print(tabulate(df, headers='keys', tablefmt='grid', showindex=False))
                else:
                    print(df.to_string(index=False))
        else:
            print("\nNo tables extracted.")



Processing PDF: Z:\HTOC\HTOC Reports\I&W Reports\3. Finished I&W Reports\HTOC-20250624-1015-A.pdf


Processing PDF: Z:\HTOC\HTOC Reports\I&W Reports\3. Finished I&W Reports\HTOC-20250625-1031-A.pdf


Processing PDF: Z:\HTOC\HTOC Reports\I&W Reports\3. Finished I&W Reports\HTOC-20250703-1249-A.pdf


Processing PDF: Z:\HTOC\HTOC Reports\I&W Reports\3. Finished I&W Reports\HTOC-20250625-1031-A.pdf


Processing PDF: Z:\HTOC\HTOC Reports\I&W Reports\3. Finished I&W Reports\HTOC-20250703-1249-A.pdf


Processing PDF: Z:\HTOC\HTOC Reports\I&W Reports\3. Finished I&W Reports\HTOC-20250707-0917-A.pdf


Processing PDF: Z:\HTOC\HTOC Reports\I&W Reports\3. Finished I&W Reports\HTOC-20250707-0917-A.pdf


--- Data from PDF: HTOC-20250624-1015-A.pdf ---

Title/Serial Number Paragraph:

HTOC—Four Possibly Malicious Dutch TOR Nodes Seen in Observations for
Seven HTOC Partners, June 2025/I&W-20250624-1015-A

Extracted 2 table(s):

Page 1 - Table 1:

Indicators/Identifiers Indicator\nType                

In [13]:
import pandas as pd
import re

# --- Extract Descriptions ---
pdf_descriptions = {}
for pdf_name, data in all_pdf_data.items():
    desc = data.get("title_serial_paragraph", "")
    match = re.search(r"[—-]\s*(.*?)(?:Seen|seen)", desc, re.DOTALL)
    if match:
        extracted = match.group(1).strip()
    else:
        dash_idx = max(desc.rfind("—"), desc.rfind("-"))
        extracted = desc[dash_idx + 1:].strip() if dash_idx != -1 else ""
    serial = pdf_name.replace("HTOC-", "").replace(".pdf", "")
    pdf_descriptions[serial] = extracted

# --- Extract Indicators ---
all_indicators = []

for pdf_name, data in all_pdf_data.items():
    serial = pdf_name.replace("HTOC-", "").replace(".pdf", "")
    for table_info in data.get("tables", []):
        df = table_info.get("dataframe")
        if df is None or df.empty:
            continue

        # Clean column headers
        df.columns = [str(col).replace('\n', ' ').strip() for col in df.columns]

        # Clean 'Indicator Type' if present
        if 'Indicator Type' in df.columns:
            df['Indicator Type'] = df['Indicator Type'].astype(str).str.replace('\n', ' ', regex=False).str.strip()

        # Merge 'Indicator' and 'Type' into 'Indicator Type' if needed
        if 'Indicator' in df.columns and 'Type' in df.columns:
            df['Indicator Type'] = df['Indicator'].fillna('') + ' ' + df['Type'].fillna('')
            df['Indicator Type'] = df['Indicator Type'].str.strip()
            df.drop(columns=['Indicator', 'Type'], inplace=True)

        # Detect relevant indicator columns
        indicator_cols = [
            col for col in df.columns
            if 'indicator' in col.lower() and (' ' not in col) and ('\n' not in col)
        ]
        if not indicator_cols:
            continue

        indicator_type_col = 'Indicator Type' if 'Indicator Type' in df.columns else None
        observed_by_col = 'Observed By' if 'Observed By' in df.columns else None

        for ind_col in indicator_cols:
            temp_df = df[[ind_col]].copy().rename(columns={ind_col: 'Indicator'})
            temp_df['Indicator'] = temp_df['Indicator'].astype(str).str.strip()

            # Add optional columns
            temp_df['Type'] = df[indicator_type_col] if indicator_type_col else pd.NA
            temp_df['Affected Partner(s)'] = df[observed_by_col].astype(str).str.replace('\n', ', ') if observed_by_col else pd.NA

            # Add I&W Serial
            temp_df['I&W Serial'] = serial

            all_indicators.append(temp_df)

# --- Combine and Clean ---
if all_indicators:
    combined_indicators_df = pd.concat(all_indicators, ignore_index=True)

    # Drop empty indicators
    combined_indicators_df = combined_indicators_df[combined_indicators_df['Indicator'] != '']

    # Add Description column
    combined_indicators_df['Description'] = combined_indicators_df['I&W Serial'].map(pdf_descriptions)

    print("Combined Indicators DataFrame:")
    display(combined_indicators_df)
else:
    print("No indicators found in stored tables.")


Combined Indicators DataFrame:


,Indicator,Type,Affected Partner(s),I&W Serial,Description
0,45.90.185[.]101,IPv4 Address,"FDA, VA, CMS, HRSA, IHS, OS",20250624-1015-A,Four Possibly Malicious Dutch TOR Nodes
1,45.90.185[.]119,IPv4 Address,"NIH, FDA, CMS, HRSA, VA, IHS, OS",20250624-1015-A,Four Possibly Malicious Dutch TOR Nodes
2,103.191.76[.]181,IPv4 Address,"CMS, NIH, VA",20250625-1031-A,Malaysian IP Address Reported by the USG as In...
3,45.9.156[.]101,IPv4 Address,"HRSA, OS, CMS, VA",20250703-1249-A,Eleven IP Addresses Reported in June 2025 by t...
4,45.90.185[.]107,IPv4 Address,"FDA, HRSA, OS, CMS, IHS, VA",20250703-1249-A,Eleven IP Addresses Reported in June 2025 by t...
5,96.9.125[.]211,IPv4 Address,"NIH, HRSA, FDA, VA, IHS, OS",20250703-1249-A,Eleven IP Addresses Reported in June 2025 by t...
6,45.90.185[.]114,IPv4 Address,"HRSA, CMS, VA, FDA, IHS, OS",20250703-1249-A,Eleven IP Addresses Reported in June 2025 by t...
7,45.191.44[.]21,,"FDA, DHA, VA",20250707-0917-A,Numerous IP Addresses Reported in June 2025 by...
8,198.98.62[.]158,,"OS, IHS, FDA",20250707-0917-A,Numerous IP Addresses Reported in June 2025 by...
9,45.90.185[.]115,,"VA, FDA, IHS, OS",20250707-0917-A,Numerous IP Addresses Reported in June 2025 by...


In [14]:
DOC_PATH = r"Z:\HTOC\HTOC Reports\Reported IOCs Tracker - Jaytlin's AutoTest.xlsx"

if 'combined_indicators_df' in locals():
    # Remove Description column before saving, if present
    df_to_save = combined_indicators_df.drop(columns=['Description'], errors='ignore')
    # Reorder columns to put 'Affected Partner(s)' at the end if present
    cols = [col for col in df_to_save.columns if col != 'Affected Partner(s)']
    if 'Affected Partner(s)' in df_to_save.columns:
        cols.append('Affected Partner(s)')
    df_to_save = df_to_save[cols]
    # Remove duplicate records before saving
    deduped_df = df_to_save.drop_duplicates()
    # Auto-adjust column width using openpyxl if available
    try:
        with pd.ExcelWriter(DOC_PATH, engine='openpyxl') as writer:
            deduped_df.to_excel(writer, index=False)
            worksheet = writer.sheets['Sheet1'] if 'Sheet1' in writer.sheets else writer.book.active
            for column_cells in worksheet.columns:
                max_length = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
                worksheet.column_dimensions[column_cells[0].column_letter].width = max_length + 2
    except ImportError:
        # Fallback if openpyxl is not installed
        deduped_df.to_excel(DOC_PATH, index=False)
    print(f"Data written to {DOC_PATH}")
else:
    print("combined_indicators_df not found.")

Data written to Z:\HTOC\HTOC Reports\Reported IOCs Tracker - Jaytlin's AutoTest.xlsx


In [ ]:
from pptx.dml.color import RGBColor
from pptx.util import Pt, Inches
from pptx import Presentation
import copy

PPTX_PATH = r'Z:\HTOC\Data_Analytics\I_W\Master\Partner Input and I_W_Slide_Template.pptx'

prs = Presentation(PPTX_PATH)

# ——— 1) PIN SLIDE 3 BEFORE DUPLICATION ———
sldIdLst = prs.slides._sldIdLst
orig_slide3_sldId = list(sldIdLst)[2] if len(sldIdLst) >= 3 else None

if len(prs.slides) < 2:
    print("Presentation does not have a slide 2.")
else:
    slide2 = prs.slides[1]
    print("Slide 2 pulled from presentation.")

    # Find the first table shape and footer on slide 2
    table_shape = None
    footer_shapes = []
    for shape in slide2.shapes:
        if shape.has_table:
            table_shape = shape
        elif shape.is_placeholder and "footer" in shape.name.lower():
            footer_shapes.append(shape)

    if table_shape is None:
        print("No table found on slide 2.")

if table_shape is None:
    print("No table found on slide 2.")
else:
    pptx_table = table_shape.table

    # Adjust table dimensions to fit within slide boundaries
    table_shape.left   = Inches(0.42)
    table_shape.top    = Inches(0.25)
    table_shape.width  = Inches(12.42)
    table_shape.height = Inches(6.73)

    # Header row is row 2 (0-based index)
    header_row = pptx_table.rows[2]
    indicator_col_idx = None
    iw_number_col_idx = None

    for idx, cell in enumerate(header_row.cells):
        txt = cell.text.lower()
        if "indicator" in txt:
            indicator_col_idx = idx
        if "i & w number" in txt:
            iw_number_col_idx = idx

    if indicator_col_idx is None:
        print("No 'Indicator' column found in the table.")
    else:
        indicators   = combined_indicators_df["Indicator"].tolist()
        iw_serials   = combined_indicators_df["I&W Serial"].tolist()
        descriptions = combined_indicators_df["Description"].tolist()

        max_rows         = len(pptx_table.rows) - 3  # available for data
        total_indicators = len(indicators)
        slides_needed    = (total_indicators + max_rows - 1) // max_rows

        for slide_idx in range(slides_needed):
            if slide_idx > 0:
                # Duplicate slide 2 for additional slides
                new_slide = prs.slides.add_slide(slide2.slide_layout)
                for shape in slide2.shapes:
                    if shape.has_table:
                        new_table_shape = copy.deepcopy(shape)
                        new_slide.shapes._spTree.insert_element_before(
                            new_table_shape._element, 'p:extLst'
                        )
                        pptx_table = new_table_shape.table
                    elif shape in footer_shapes:
                        new_footer_shape = copy.deepcopy(shape)
                        new_slide.shapes._spTree.insert_element_before(
                            new_footer_shape._element, 'p:extLst'
                        )

                # Adjust table dimensions for new slide
                new_table_shape.left   = Inches(0.42)
                new_table_shape.top    = Inches(0.25)
                new_table_shape.width  = Inches(12.42)
                new_table_shape.height = Inches(6.73)

            start_idx = slide_idx * max_rows
            end_idx   = min(start_idx + max_rows, total_indicators)

            for i in range(start_idx, end_idx):
                row_idx = 3 + (i - start_idx)  # Data starts at row 3
                if row_idx >= len(pptx_table.rows):
                    break

                # Write Indicator
                cell = pptx_table.cell(row_idx, indicator_col_idx)
                cell.text = str(indicators[i])
                for paragraph in cell.text_frame.paragraphs:
                    for run in paragraph.runs:
                        run.font.color.rgb = RGBColor(0, 0, 0)
                        run.font.size      = Pt(9)
                        run.font.name      = "Arial"

                # Write I&W Serial if column exists
                if iw_number_col_idx is not None:
                    iw_serial = iw_serials[i] if i < len(iw_serials) else ""
                    cell = pptx_table.cell(row_idx, iw_number_col_idx)
                    cell.text = f"HTOC-{iw_serial}" if iw_serial else ""
                    for paragraph in cell.text_frame.paragraphs:
                        for run in paragraph.runs:
                            run.font.color.rgb = RGBColor(0, 0, 0)
                            run.font.size      = Pt(9)
                            run.font.name      = "Arial"

                # Write Description if column exists
                desc_col_idx = None
                for idx, hcell in enumerate(header_row.cells):
                    if "description" in hcell.text.lower():
                        desc_col_idx = idx
                        break
                if desc_col_idx is not None:
                    cell = pptx_table.cell(row_idx, desc_col_idx)
                    cell.text = str(descriptions[i])
                    for paragraph in cell.text_frame.paragraphs:
                        for run in paragraph.runs:
                            run.font.color.rgb = RGBColor(0, 0, 0)
                            run.font.size      = Pt(9)
                            run.font.name      = "Arial"

            # Add text from original slide 2 to duplicates
            textbox = new_slide.shapes.add_textbox(Inches(0.03), Inches(7.1), Inches(12), Inches(1))
            text_frame = textbox.text_frame
            text_frame.text = (
                "Please enter an X in the appropriate column.  "
                "Indicator Disposition Description on next Slide"
            )
            for paragraph in text_frame.paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(16)
                    run.font.name = "Arial"

        # ——— 2) RE-APPEND SLIDE 3 AT THE END ———
        if orig_slide3_sldId is not None:
            sldIdLst.remove(orig_slide3_sldId)
            sldIdLst.append(orig_slide3_sldId)

    new_pptx_path = PPTX_PATH.replace('.pptx', '_with_indicators.pptx')
    prs.save(new_pptx_path)
    print(f"Saved new presentation with indicators to: {new_pptx_path}")


Slide 2 pulled from presentation.
Saved new presentation with indicators to: Z:\HTOC\Data_Analytics\I_W\Master\Partner Input and I_W_Slide_Template_with_indicators.pptx
